# 🌳 Composite Design Pattern — Hinglish Beginner Guide

**Composite Design Pattern** ek **structural design pattern** hai jo objects ko 
**tree-like (hierarchical) structure** mein organize karta hai, aur Client ko 
**single object** aur **group of objects** dono ko **SAME tarike se treat** karne deta hai.

> **Simple bhasha mein:** Socho tumhare computer mein ek **folder** hai jisme kuch **files** 
> hain, aur kuch **sub-folders** bhi hain (jinme aur files/folders ho sakte hain). Ab agar 
> tum "ye folder delete karo" bolte ho, to system automatically andar ki SAB files aur 
> sub-folders bhi delete kar deta hai — chahe wo kितni bhi deep nested ho. Tumhe har file 
> aur folder ke liye **alag se** delete command nahi dena padta — sabko **same "delete" 
> operation** se handle kar liya jata hai. Yahi **Composite Pattern** ka core idea hai — 
> **"Single item ho ya poora group, dono ko same tarike se treat karo."**

Is notebook mein hum seekhenge:
1. Composite Pattern kya hai aur kyun use karte hain
2. **Bad/Without Pattern approach** — problem kya hai wo dekhenge
3. Text-based UML diagrams (comments ke andar)
4. **Best/With Pattern approach** — step-by-step Python implementation (File System example)
5. Part-Whole Hierarchy concept
6. Advantages/Disadvantages
7. **5 Practice Questions with Answers** (real-world examples ke saath)


## 📁 Real-Life Analogy (Samjhne ke liye)

**File System** — sabse common example:

```
📁 My Documents (Directory)
   📄 document.txt (File)
   📄 image.jpg (File)
   📁 Projects (Directory)
      📄 code.py (File)
      📁 Backup (Directory)
         📄 old_code.py (File)
```

Yaha do tarah ki cheezein hain:
- **File** — akela, individual item (kuch aur contain nahi karta) → isko **Leaf** kehte hain
- **Directory** — group jisme files AUR aur directories dono ho sakte hain → isko **Composite** kehte hain

Ab agar tumhe **poore "My Documents" folder ka size dikhana ho** ya **display karna ho**, to 
tum nahi chahoge ki alag-alag logic likhna pade — ek file ke liye alag, directory ke liye alag, 
aur nested directory ke liye aur alag!

**Composite Pattern isko solve karta hai** — File aur Directory dono ek **common interface** 
(`FileSystemComponent`) implement karte hain jisme `display()` method hota hai. Jab tum 
Directory pe `display()` call karte ho, wo **recursively** apne andar ke sab components 
(files + sub-directories) pe bhi `display()` call kar deta hai — automatically!

**Doosre real-world examples:**
- 🎨 **Graphics/GUI Libraries**: Shapes (circle, rectangle) ko group karke complex shapes (smiley face) banana
- 🏢 **Organization Structure**: Employee (individual) aur Department (group of employees/sub-departments)
- 🌲 **Menu Systems**: Menu items aur Sub-menus jo aur menu items rakh sakte hain


## ❌ Step 1: Problem — BINA Composite Pattern ke (Bad Approach)

Chalo pehle dekhte hain agar hum **naive tarike** se File aur Directory ko **alag-alag 
handle** karein (koi common interface na ho), to kya dikkat aati hai.


In [ ]:
"""
BAD APPROACH — File aur Directory ko ALAG-ALAG classes ki tarah treat karna
==================================================================================
Idea: File aur Directory mein koi COMMON interface nahi hai. Client ko
har jagah pata hona chahiye ki wo File se deal kar raha hai ya Directory se.
"""

class File:
    def __init__(self, name, size):
        self.name = name
        self.size = size

    def show_file(self):
        print(f"File: {self.name} ({self.size} bytes)")


class Directory:
    def __init__(self, name):
        self.name = name
        self.files = []          # sirf files store kar sakta hai
        self.subdirectories = []  # sub-directories ALAG list mein

    def add_file(self, file):
        self.files.append(file)

    def add_subdirectory(self, directory):
        self.subdirectories.append(directory)

    def show_directory(self):
        print(f"Directory: {self.name}")
        # ❌ Client ko manually check karna padega -- files ALAG loop,
        #    subdirectories ALAG loop -- aur subdirectory ke andar bhi
        #    yahi dobara karna padega (messy recursive logic client mein)
        for f in self.files:
            f.show_file()
        for sub in self.subdirectories:
            sub.show_directory()   # naam bhi ALAG method hai (show_file vs show_directory)


# Client -- File aur Directory ko ALAG-ALAG treat karna pad raha hai
file1 = File("document.txt", 1024)
file2 = File("image.jpg", 2048)

directory = Directory("My Documents")
directory.add_file(file1)     # ❌ files ke liye ALAG method
directory.add_file(file2)

sub_dir = Directory("Projects")
sub_dir.add_file(File("code.py", 512))
directory.add_subdirectory(sub_dir)   # ❌ directories ke liye ALAG method

directory.show_directory()

"""
❌ PROBLEMS WITH THIS BAD APPROACH:

1. NO COMMON INTERFACE: File aur Directory ke method names ALAG hain
   (show_file() vs show_directory()) -- Client ko HAMESHA yaad rakhna
   padta hai ki kaunse object pe kaunsa method call karna hai.

2. SEPARATE STORAGE LOGIC: Directory ke andar files aur subdirectories
   ke liye ALAG-ALAG lists (self.files, self.subdirectories) rakhni
   padi -- agar kal ek naya type (jaise "Shortcut") add karna ho, to
   EK AUR list aur EK AUR method banana padega.

3. CLIENT CODE COMPLEX: Client ko pata hona chahiye ki "isse add_file()
   se add karna hai ya add_subdirectory() se" -- ye TYPE-CHECKING wali
   mentality hai, jo Composite Pattern avoid karna chahta hai.

4. HARD TO EXTEND: Naya component type (jaise "SymbolicLink") add karna
   ho to Directory class mein NAYI list + NAYA method + NAYA loop
   sab add karna padega.

LESSON: Humein ek SINGLE common interface chahiye jisse File aur
Directory dono SAME tarike se treat ho sakein, aur Directory ek hi
list mein kisi bhi type ka component store kar sake. Yahi Composite
Pattern solve karta hai! Chalo best approach dekhte hain 👇
"""


In [ ]:
"""
TEXT-BASED UML CLASS DIAGRAM — Composite Design Pattern (BEST APPROACH)
================================================================================

                    +--------------------------------+
                    |       FileSystemComponent           |
                    |         (Component - ABC)             |
                    +--------------------------------+
                    | + display()  (abstract)               |
                    +----------------^-------------------+
                                     |
                     +---------------+-----------------------+
                     |                                        |
          +----------+---------+                +-------------+-------------+
          |        File            |                |         Directory              |
          |        (Leaf)           |                |        (Composite)              |
          +------------------------+                +--------------------------------+
          | - name : str              |                | - name : str                       |
          | - size : int                 |               | - components : list[Component]      |
          +------------------------+                +--------------------------------+
          | + display()                    |          | + display()                            |
          |   -> "File: name (size bytes)"  |          |   -> print name, then LOOP over            |
          +------------------------+        |          |      components calling display() on each  |
                                              |          | + add_component(c)                            |
                                              |          | + remove_component(c)                          |
                                              |          +--------------------------------+
                                              |                        |
                                              |                        | can CONTAIN
                                              |                        v
                                              |            +----------------------+
                                              +----------->|  File  OR  Directory    |  <-- RECURSIVE!
                                                           |  (any Component)          |     Directory can
                                                           +----------------------+     hold MORE Directories


TREE STRUCTURE (Part-Whole Hierarchy) — File System Example
====================================================================

                          Directory("My Documents")
                         /              |              \
                   File(doc.txt)   File(image.jpg)   Directory("Projects")
                                                        /              \
                                                  File(code.py)   Directory("Backup")
                                                                        |
                                                                  File(old_code.py)

  Jab "My Documents".display() call hota hai, ye RECURSIVELY andar ke
  saare components (files + nested directories) pe bhi display() call
  kar deta hai -- chahe hierarchy kितni bhi deep ho!


FLOW DIAGRAM — display() call kaise recursively chalta hai
====================================================================

    Client                Directory("My Docs")      Directory("Projects")     File
      |                          |                          |                   |
      |--- directory.display() ->|                          |                   |
      |                          |--- print "My Docs"          |                |
      |                          |--- file1.display() -------------------------->|
      |                          |                          |                   |--- print "document.txt"
      |                          |--- sub_dir.display() --->|                   |
      |                          |                          |--- print "Projects" |
      |                          |                          |--- child.display() ->|
      |                          |                          |                   |--- print "code.py"
"""

print("UML aur tree diagram upar comment mein hai — dhyan se padho! 👆")


## 🧩 4 Main Components (Yaad rakhne wali cheez)

| Component | Kaam kya hai | Analogy (File System example) |
|---|---|---|
| **Component (Interface)** | Common interface jo Leaf aur Composite dono implement karte hain | `FileSystemComponent` (jisme `display()` hai) |
| **Leaf** | Individual object, koi children nahi hote | `File` |
| **Composite** | Container object jo Leaf AUR doosre Composite dono rakh sakta hai | `Directory` |
| **Client** | Component interface use karta hai, Leaf/Composite mein farak nahi karta | Client code jo `display()` call karta hai |

### Working — 3 Steps
1. **Client** sirf `Component` interface (`display()`) ke through kaam karta hai — usko pata 
   nahi (ya matlab nahi) ki wo File se baat kar raha hai ya Directory se
2. **Leaf** apna kaam khud karta hai (jaise File apna naam/size print kar deta hai)
3. **Composite** operation ko apne **sab children pe recursively forward** kar deta hai


## 🧩 Part-Whole (Whole-Part) Hierarchy — Ye Concept Kya Hai?

Composite Pattern ka core idea **"Part-Whole hierarchy"** pe based hai:

- **Parts** = chhote, individual objects (jaise circle, rectangle, ek single file)
- **Whole** = bade, complex objects jo parts se milke bante hain (jaise smiley face jo circles + rectangles se bana hai, ya ek folder jo files se bana hai)

**Graphic Design Example:**
```
Smiley Face (Whole/Composite)
   ├── Circle (Part/Leaf) -- face outline
   ├── Circle (Part/Leaf) -- left eye
   ├── Circle (Part/Leaf) -- right eye
   └── Arc (Part/Leaf)    -- smile
```

Composite Pattern ki khaasiyat ye hai ki tum **Smiley Face pe wahi operations** (jaise `draw()`, 
`move()`, `resize()`) call kar sakte ho jo tum ek **single Circle** pe karte — aur system 
automatically andar ke sab parts pe wo operation apply kar deta hai!


## ✅ Step 2: Solution — WITH Composite Pattern (Best Approach)

Ab dekhte hain **best approach** — Composite Pattern use karke File System ko 
**clean aur uniform** tareeke se implement karte hain.

### Implementation Steps:
1. **Component Interface** banao (FileSystemComponent) — common contract
2. **Leaf** banao (File) — individual object, no children
3. **Composite** banao (Directory) — container, ek hi list mein kisi bhi Component type ko rakhe
4. **Client** — sirf Component interface se kaam kare, File/Directory mein farak na kare


In [ ]:
"""
STEP 1: COMPONENT INTERFACE
=================================
Ye common contract hai jo LEAF (File) aur COMPOSITE (Directory) dono
follow karenge -- isi wajah se dono ko UNIFORMLY treat kar payenge.
"""

from abc import ABC, abstractmethod

class FileSystemComponent(ABC):
    @abstractmethod
    def display(self):
        pass


In [ ]:
"""
STEP 2: LEAF (File)
=========================
Ye individual object hai -- iske koi children nahi hote.
Ye sirf apna kaam khud karta hai.
"""

class File(FileSystemComponent):
    def __init__(self, name, size):
        self.name = name
        self.size = size

    def display(self):
        print(f"File: {self.name} ({self.size} bytes)")


In [ ]:
"""
STEP 3: COMPOSITE (Directory)
===================================
Ye ek CONTAINER hai -- iske andar File objects AUR doosre Directory
objects (RECURSIVE!) dono aa sakte hain, EK HI list mein.

Jab display() call hota hai, ye khud ka naam print karta hai, phir
apne HAR child pe display() call karta hai -- chahe wo child File ho
ya Directory, usko FARAK NAHI PADTA (dono FileSystemComponent hain)!
"""

class Directory(FileSystemComponent):
    def __init__(self, name):
        self.name = name
        self.components = []   # <-- EK HI list, File aur Directory dono ke liye!

    def add_component(self, component):
        self.components.append(component)

    def remove_component(self, component):
        self.components.remove(component)

    def display(self):
        print(f"Directory: {self.name}")
        for component in self.components:
            component.display()   # <-- RECURSIVE call, File ho ya Directory, same call!


In [ ]:
"""
STEP 4: CLIENT CODE
=========================
Client sirf display() call karta hai -- usko FARAK NAHI PADTA ki
kaunsa object File hai ya Directory, sab kuch UNIFORMLY handle ho jata hai.
"""

def main():
    # Leaf objects (files)
    file1 = File("document.txt", 1024)
    file2 = File("image.jpg", 2048)

    # Composite object (directory)
    my_documents = Directory("My Documents")
    my_documents.add_component(file1)
    my_documents.add_component(file2)

    # Nested composite -- Directory ke andar Directory!
    projects = Directory("Projects")
    projects.add_component(File("code.py", 512))

    backup = Directory("Backup")
    backup.add_component(File("old_code.py", 256))
    projects.add_component(backup)      # Directory ke andar Directory!

    my_documents.add_component(projects)  # aur ye bhi My Documents ke andar!

    # Client sirf EK method call karta hai -- poora tree recursively display ho jata hai!
    my_documents.display()


main()

"""
EXPLANATION — Ye BEST APPROACH kyun better hai bad approach se:

✅ UNIFORM TREATMENT: Client sirf "component.display()" call karta hai --
   File ho ya Directory, dono ka method naam SAME hai (display()).

✅ SINGLE STORAGE LIST: Directory ke andar files aur subdirectories dono
   EK HI list (self.components) mein store hote hain -- alag-alag lists
   ki zarurat nahi.

✅ AUTOMATIC RECURSION: "My Documents".display() call karte hi POORA
   nested tree (chahe kितna bhi deep ho) automatically display ho jata
   hai -- Client ko manually recursion likhne ki zarurat nahi.

✅ EASY TO EXTEND: Naya component type (jaise "SymbolicLink") add karna
   ho, bas FileSystemComponent implement karo aur usko kisi Directory
   mein add_component() se daal do -- purana code bilkul safe rahega.

Yahi Composite Pattern ki asli power hai — "single item ho ya poora
tree, sabko SAME tarike se treat karo"!
"""


## 🌍 Real-World Applications

| Application | Kaise use hota hai |
|---|---|
| 📁 **File Systems** | Files (leaf) aur Directories (composite) ek common interface se manage hote hain |
| 🎨 **Graphics/GUI Libraries** | Shapes ko group karke complex drawings banana (smiley face, diagrams) |
| 🏢 **Organization Structures** | Employee (leaf) aur Department (composite jisme employees + sub-departments hon) |
| 🍔 **Menu Systems** | Menu items aur Sub-menus (jo aur menu items rakh sakte hain) |
| 🖥️ **UI Component Trees** | Ek Window mein Panels, aur Panels mein Buttons/Labels — sab ek common "render()" interface follow karte hain |


## ✅ Advantages (Fayde)

| # | Advantage | Explanation |
|---|---|---|
| 1 | Hierarchical Structure | Tree-like structures ko naturally represent kar sakte ho |
| 2 | Simplified Client Code | Client ko single vs composite object mein farak nahi karna padta |
| 3 | Flexibility | Objects add/remove karna aasan hai, client code affect nahi hota |
| 4 | Code Reusability | Same operation (jaise display()) parts aur wholes dono pe kaam karta hai |

## ❌ Disadvantages (Nuksan)

| # | Disadvantage | Explanation |
|---|---|---|
| 1 | Complex Implementation | Sab objects ke liye common interface design karna thoda tricky ho sakta hai |
| 2 | Performance Overhead | Bahut deep hierarchies traverse karna slow ho sakta hai |
| 3 | Limited Type Safety | Common interface ki wajah se kabhi kabhi invalid operations bhi allow ho sakte hain (jaise File pe add_component() call karne ki koshish) |
| 4 | Extra Memory Usage | Har composite ko apne children ka reference store karna padta hai |


---
# 📝 Practice Questions (5 Real-World Examples)

Har question mein: **Problem Statement → Try Yourself → Solution Code → Explanation**

Beginners: pehle khud try karo, phir solution dekho! 💪


## Q1. 🏢 Organization Structure (Employee + Department)

**Problem:** Ek company ka org-chart banao jisme `Employee` (leaf) aur `Department` 
(composite, jisme employees AUR sub-departments dono ho sakte hain) ho. Ek `show_details()` 
method ho jo poora hierarchy print kare, aur total salary bhi calculate kare.

**Try yourself first**, phir neeche solution dekho.


In [ ]:
"""
SOLUTION — Organization Structure
========================================
"""

from abc import ABC, abstractmethod

# Component
class OrgComponent(ABC):
    @abstractmethod
    def show_details(self, indent=0):
        pass
    @abstractmethod
    def get_total_salary(self):
        pass

# Leaf
class Employee(OrgComponent):
    def __init__(self, name, role, salary):
        self.name = name
        self.role = role
        self.salary = salary

    def show_details(self, indent=0):
        print(" " * indent + f"👤 {self.name} ({self.role}) - Rs.{self.salary}")

    def get_total_salary(self):
        return self.salary

# Composite
class Department(OrgComponent):
    def __init__(self, name):
        self.name = name
        self.members = []   # Employee AUR Department dono aa sakte hain

    def add_member(self, member):
        self.members.append(member)

    def show_details(self, indent=0):
        print(" " * indent + f"🏢 {self.name}")
        for member in self.members:
            member.show_details(indent + 4)   # recursive call, indent badhta jayega

    def get_total_salary(self):
        return sum(member.get_total_salary() for member in self.members)


# Client
engineering = Department("Engineering")
engineering.add_member(Employee("Rohit", "Senior Engineer", 90000))
engineering.add_member(Employee("Priya", "Engineer", 70000))

backend_team = Department("Backend Team")
backend_team.add_member(Employee("Aman", "Backend Dev", 65000))
engineering.add_member(backend_team)   # Department ke andar Department!

engineering.show_details()
print(f"\n💰 Total Salary Budget: Rs.{engineering.get_total_salary()}")

"""
EXPLANATION:
Employee (Leaf) aur Department (Composite) dono "get_total_salary()" aur
"show_details()" implement karte hain. Jab humne "Engineering".get_total_salary()
call kiya, ye khud recursively "Backend Team" (jo khud ek Department hai)
ka bhi salary sum kar leta hai -- Client ko manual recursion nahi likhni padi!
"""


## Q2. 🛒 Shopping Cart with Product Bundles

**Problem:** Ek E-commerce cart banao jisme `Product` (leaf, individual item) aur 
`Bundle` (composite, jisme products AUR sub-bundles ho sakte hain, jaise "Combo Offer") 
ho. Total price calculate karne wala method banao.


In [ ]:
"""
SOLUTION — Shopping Cart with Bundles
=============================================
"""

from abc import ABC, abstractmethod

# Component
class CartItem(ABC):
    @abstractmethod
    def get_price(self):
        pass
    @abstractmethod
    def show(self, indent=0):
        pass

# Leaf
class Product(CartItem):
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def get_price(self):
        return self.price

    def show(self, indent=0):
        print(" " * indent + f"📦 {self.name} - Rs.{self.price}")

# Composite
class Bundle(CartItem):
    def __init__(self, name):
        self.name = name
        self.items = []

    def add_item(self, item):
        self.items.append(item)

    def get_price(self):
        return sum(item.get_price() for item in self.items)

    def show(self, indent=0):
        print(" " * indent + f"🎁 {self.name} (Bundle)")
        for item in self.items:
            item.show(indent + 4)


# Client
laptop = Product("Laptop", 55000)
mouse = Product("Wireless Mouse", 800)
bag = Product("Laptop Bag", 1200)

combo = Bundle("Student Combo Offer")
combo.add_item(laptop)
combo.add_item(mouse)
combo.add_item(bag)

headphones = Product("Headphones", 2000)

cart = Bundle("My Cart")
cart.add_item(combo)         # Bundle ke andar Bundle!
cart.add_item(headphones)    # aur ek standalone product bhi

cart.show()
print(f"\n💰 Total Cart Price: Rs.{cart.get_price()}")

"""
EXPLANATION:
"My Cart" ek Bundle hai jisme "Student Combo Offer" (jo khud ek Bundle
hai, 3 products ke saath) AUR ek standalone "Headphones" product dono
hain. get_price() call karte hi POORA nested structure recursively add
ho jata hai -- Client ko pata bhi nahi chalta ki andar kितne levels
ki nesting hai.
"""


## Q3. 🎨 Graphics — Smiley Face from Shapes

**Problem:** Document mein "Smiley Face" ka example diya gaya tha — Circle aur Arc shapes 
se milkar bana hua. Composite Pattern use karke ek `Shape` interface banao jisme 
`draw()` ho, aur `Circle`/`Arc` (leaf) se ek `Group` (composite) banake "Smiley Face" banao.


In [ ]:
"""
SOLUTION — Graphics Composite (Smiley Face)
===================================================
"""

from abc import ABC, abstractmethod

# Component
class Shape(ABC):
    @abstractmethod
    def draw(self):
        pass

# Leaf
class Circle(Shape):
    def __init__(self, label):
        self.label = label
    def draw(self):
        print(f"⭕ Drawing Circle: {self.label}")

class Arc(Shape):
    def __init__(self, label):
        self.label = label
    def draw(self):
        print(f"〰️ Drawing Arc: {self.label}")

# Composite
class Group(Shape):
    def __init__(self, name):
        self.name = name
        self.shapes = []

    def add_shape(self, shape):
        self.shapes.append(shape)

    def draw(self):
        print(f"--- Drawing Group: {self.name} ---")
        for shape in self.shapes:
            shape.draw()


# Client — Smiley Face banate hain shapes ko combine karke
smiley_face = Group("Smiley Face")
smiley_face.add_shape(Circle("Face Outline"))
smiley_face.add_shape(Circle("Left Eye"))
smiley_face.add_shape(Circle("Right Eye"))
smiley_face.add_shape(Arc("Smile"))

# Ek Smiley ko doosre bade Group mein bhi daal sakte hain! (Part-Whole hierarchy)
poster = Group("Birthday Poster")
poster.add_shape(smiley_face)     # Group ke andar Group!
poster.add_shape(Circle("Balloon"))

poster.draw()

"""
EXPLANATION:
"Smiley Face" khud ek Group (Composite) hai jo 3 Circles + 1 Arc se bana
hai. Is Smiley Face ko humne ek BADE Group ("Birthday Poster") mein bhi
daal diya — bilkul waise hi jaise ek Directory doosri Directory ke andar
ja sakti hai. Ye "Part-Whole hierarchy" ka perfect real-world example hai
jahan chhote parts (Circle, Arc) milke wholes (Smiley Face) banate hain,
aur wholes khud kisi aur bade whole ka part ban sakte hain.
"""


## Q4. 🤔 Conceptual Question (No Code)

**Question:** Composite Pattern mein bhi to Composite object apne children ko "wrap/hold" 
karta hai — aur Decorator Pattern mein bhi ek object doosre ko wrap karta hai. *"Ye dono 
same lag rahe hain, farak kya hai?"*

### ✅ Answer:

| Aspect | Decorator Pattern | Composite Pattern |
|---|---|---|
| **Purpose** | Ek SINGLE object pe naya **behavior/feature** add karna | Objects ko **tree/hierarchy** mein organize karke UNIFORM treat karna |
| **Number of wrapped objects** | Hamesha **EK** object ko wrap karta hai (linear chain) | **MULTIPLE** children ho sakte hain (tree structure, ek se zyada branches) |
| **Structure** | Linear: `A(B(C(D)))` — ek seedhi chain | Tree: ek Composite ke MULTIPLE children ho sakte hain, jo khud bhi Composite ho sakte hain |
| **Kab use karo** | Jab ek object mein progressively features add karne hon | Jab **part-whole hierarchy** represent karni ho (files/folders, employees/departments) |
| **Example** | `SugarDecorator(MilkDecorator(PlainCoffee()))` — coffee pe layers | `Directory` jisme MULTIPLE files aur MULTIPLE sub-directories ek saath hon |

**Simple yaad rakhne ka tarika:** Decorator = **"Ek line mein wraps"** (jaise pyaaz ke layers, 
ek ke upar ek). Composite = **"Ek tree jisme branches"** (jaise ek folder jisme MULTIPLE 
files aur folders ek saath hote hain, har branch alag).

**Extra Insight:** Dono patterns ek **recursive composition** use karte hain, lekin 
Decorator ka goal hai **"enhance karna"**, jabki Composite ka goal hai 
**"group ko individual jaisa treat karna"**.


## Q5. 🧪 Debug Karo: Ye Design Kyun Problematic Hai?

**Problem:** Ek naya developer ek **Menu System** banana chahta hai (jaise restaurant app 
mein categories aur items). Usne neeche wala approach try kiya jisme `Category` aur `MenuItem` 
completely alag-alag handle hote hain, koi common interface nahi. Pehle problem identify 
karo, phir Composite Pattern se fix karo.


In [ ]:
"""
❌ PROBLEMATIC CODE (Try to spot the issues before scrolling to solution!)
================================================================================

class MenuItem:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def show_item(self):
        print(f"{self.name} - Rs.{self.price}")


class Category:
    def __init__(self, name):
        self.name = name
        self.items = []
        self.subcategories = []   # ❌ ALAG list sub-categories ke liye

    def add_item(self, item):
        self.items.append(item)

    def add_subcategory(self, subcat):
        self.subcategories.append(subcat)

    def show_category(self):
        print(f"Category: {self.name}")
        for item in self.items:
            item.show_item()               # ❌ ALAG method naam
        for sub in self.subcategories:
            sub.show_category()             # ❌ recursive call, but manually 2 loops chahiye

# Problem: Agar "Combo Meal" jaisa naya type add karna ho (jo items + is
# apna price bhi ho), to Category class mein EK AUR list aur EK AUR loop
# banana padega. Ye scale nahi karta!
"""

# -----------------------------------------------------------------
# ✅ BEST APPROACH — Composite Pattern se FIX kiya gaya
# -----------------------------------------------------------------

from abc import ABC, abstractmethod

# Component
class MenuComponent(ABC):
    @abstractmethod
    def show(self, indent=0):
        pass

# Leaf
class MenuItem(MenuComponent):
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def show(self, indent=0):
        print(" " * indent + f"🍽️ {self.name} - Rs.{self.price}")

# Composite
class Category(MenuComponent):
    def __init__(self, name):
        self.name = name
        self.components = []   # ✅ EK HI list, MenuItem aur Category dono ke liye

    def add_component(self, component):
        self.components.append(component)

    def show(self, indent=0):
        print(" " * indent + f"📂 {self.name}")
        for component in self.components:
            component.show(indent + 4)


# Client
starters = Category("Starters")
starters.add_component(MenuItem("Spring Rolls", 150))
starters.add_component(MenuItem("Soup", 120))

main_course = Category("Main Course")
main_course.add_component(MenuItem("Butter Chicken", 280))

north_indian = Category("North Indian")
north_indian.add_component(main_course)   # ✅ Category ke andar Category!
north_indian.add_component(MenuItem("Naan", 40))

full_menu = Category("Full Menu")
full_menu.add_component(starters)
full_menu.add_component(north_indian)

full_menu.show()

"""
EXPLANATION — Bad code mein PROBLEM kya thi:
1. NO COMMON INTERFACE: MenuItem aur Category ke methods ALAG naam ke
   the (show_item() vs show_category()) -- Client ko pata hona padta
   tha ki kaunse object pe kaunsa method call karna hai.
2. SEPARATE LISTS: Category mein items aur subcategories ke liye ALAG
   lists thi -- naya type add karna scale nahi karta tha.
3. MANUAL DOUBLE LOOP: show_category() mein DO alag loops likhne pad
   rahe the, jo error-prone aur repetitive tha.

BEST APPROACH mein FIX kaise hua:
- MenuComponent common interface bana di jo MenuItem AUR Category
  dono implement karte hain
- Category ab EK HI list (self.components) mein kisi bhi
  MenuComponent (item ya category) ko store karta hai
- show() method har child pe UNIFORMLY call hota hai -- chahe wo
  MenuItem ho ya nested Category, ek hi loop se kaam ho jata hai ✅
"""


---
## 🎯 Quick Summary (Revision ke liye)

- **Composite Pattern** objects ko **tree structure** mein organize karta hai
- 4 components: **Component (Interface), Leaf, Composite, Client**
- **Part-Whole hierarchy**: chhote parts milke bade wholes banate hain, aur dono ko SAME 
  interface se treat kiya jata hai
- **Bad approach** mein Leaf aur Composite ke ALAG method names/storage hote hain (manual handling)
- **Best approach** mein ek common interface + ek hi storage list se **recursive, uniform** treatment milta hai

### Composite vs Decorator (Quick Recall)
| Pattern | Structure | Purpose |
|---|---|---|
| **Decorator** | LINEAR chain (ek object ko wrap karta hai) | Object pe naya BEHAVIOR add karna |
| **Composite** | TREE structure (multiple children ho sakte hain) | Part-whole HIERARCHY ko uniformly treat karna |

### All Patterns So Far (Quick Recall)
| Pattern | Type | Focus |
|---|---|---|
| **Singleton** | Creational | Sirf EK instance guarantee karna |
| **Builder** | Creational | Complex object ko STEP-BY-STEP banana |
| **Factory Method** | Creational | Object creation SUBCLASS ko delegate karna |
| **Decorator** | Structural | Object mein RUNTIME pe naya BEHAVIOR add karna |
| **Adapter** | Structural | Do INCOMPATIBLE interfaces ko compatible banana |
| **Facade** | Structural | COMPLEX subsystem ko SIMPLE interface ke peeche chhupana |
| **Composite** | Structural | Objects ko TREE (part-whole) hierarchy mein UNIFORMLY treat karna |

### Real projects mein kaha milega?
- File systems (files aur folders)
- Graphics/GUI libraries (shapes, UI component trees)
- Organization charts (employees aur departments)
- Menu systems (categories aur items)
- XML/HTML DOM trees (elements jo aur elements contain karte hain)

Happy Learning! 🚀
